# Тема 10 — Multi-GPU inference: tensor parallelism — масштабируем кухню по осям

**Что вы уже имеете.** Замороженный профиль движка (Тема 07), честное окно выдачи
(Тема 08) и неизменяемый образ (Тема 09). Всё это жило на **одном** GPU.

**Вопрос дня.** Одна плита заканчивается дважды: памятью (веса + KV не влезают)
и полом (шаг токена не ускоряется бесконечно). Что если нести рояль вчетвером —
но строго в ногу?

**Лаба за 90 минут (CPU, всё исполняется везде):**
1. Заморозить план: предсказать поведение кривой `t(tp)` и минимальную
   подходящую степень ДО всякого свипа (`tp-plan.json`).
2. Доказать механику: разбить матрицы по GPU (NumPy-шарды) и убедиться,
   что разбитое равно целому с точностью до float.
3. Свип степеней 1–8 по объявленной модели и честный вердикт против плана.
4. Ведомость памяти: какая степень влезает в бюджет и почему.
5. Поломки как данные: comm-bound и straggler — разбор на стенограммах.

**Честность темы.** Механика разбивки — настоящая (NumPy, до 1e-10). Все
миллисекунды — объявленная арифметика с именованными ручками; ни одно число
не цитируется как измерение GPU. Реальный vLLM с `tensor_parallel_size≥2` —
демонстрация владельца с отдельной записью.

In [ ]:
import json, os, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROFILE = 'sim'  # объявленная симуляция: механика реальна, секунды — арифметика
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
os.makedirs(RESULTS, exist_ok=True)
SEED = 2026
print(f'profile={PROFILE}  журнал={RESULTS}  seed={SEED} — всё исполняется на CPU')

## Блок 0 — Две стены одного GPU (механизм до формул)

Одна плита заканчивается дважды:

- **Стена памяти.** Веса 8B-модели в fp16 — это ~16 ГБ, плюс KV-кэш и активации —
  в те же 15 ГБ бюджета одного ускорителя. Не влезло — сервис не начался.
- **Пол шага.** Даже когда влезает: авторегрессия — последовательная цепочка;
  один поток на одном устройстве упирается в пол латентности.

Tensor parallelism (TP) отвечает на оба: **режем веса на части** между GPU,
каждый несёт свою долю, а шаг делается только **в ногу** (all-reduce).

Вопрос до формул: если_compute-часть шага падает ~1/tp, а синхронизация — нет,
то где предел «бери больше GPU»? Заморозим ответ ДО свипа.

In [ ]:
# Объявленные ручки темы: именованные константы, никакой скрытой измеренности.
# Это TEACHING-grade арифметика (как мок T07/T08): структура честная, числа учебные.
MODEL_DECL = dict(
    params_b=8e9,        # параметров в модели (8B — учебный соразмерный пример)
    bytes_per_param=2,   # fp16: 2 байта на параметр
    kv_gb=4.0,           # KV-кэш на полном батче (объявлено)
    fixed_gb=1.5,        # активации + framework overhead на GPU — НЕ делится по tp
    budget_gb=15.0,      # полезный бюджет одного ускорителя (объявлено)
)
STEP_DECL = dict(
    compute_ms=800.0,    # вычислительная часть шага на ОДНОМ GPU
    bytes_term_ms=120.0, # байтовая доля all-reduce при (tp-1)/tp
    beat_ms=25.0,        # цена ОДНОГО такта синхронизации (латентность)
    beats_per_block=2,   # all-reduce на трансформер-блок (см. фигуру t10-tp-split)
)
print('ручки объявлены: 8B fp16 = 16 ГБ весов; шаг: 800 мс compute + 120·(tp-1)/tp + 25·(tp-1) мс')

## План до свипа: замораживаем предсказание

Та же дисциплина, что прогноз колена в Теме 06 и план сборки в Теме 09:
**предсказание пишется до измерения**, иначе это воспоминание.

Два числа, которые вы заморозите в `tp-plan.json`:

1. **Колено «чистой» модели** — `t(p) = C/p + B·(p−1)/p` (БЕЗ цены такта и
   straggler'ов). Посчитайте сами: монотонна ли она? где её минимум на p∈1..8?
2. **Минимальная подходящая степень** из ведомости памяти:
   `16/p + 4/p + 1.5 ≤ 15`. Сколько нужно GPU, чтобы хотя бы влезть?

Свип позже покажет **наблюдаемое** колено по полной модели (с ценой такта и
straggler'ами). Расхождение — это механизм, который вы забыли учесть, а не шум.

In [ ]:
def plan_tp_knee(compute_ms: float, bytes_term_ms: float) -> dict:
    """Freeze the TP plan BEFORE any sweep: knee prediction and the fit degree.

    Parameters
    ----------
    compute_ms : float
        Declared compute part of one step on a single GPU.
    bytes_term_ms : float
        Declared all-reduce byte share at (tp-1)/tp.

    Returns
    -------
    dict
        predicted_knee_no_beat, fit_degree, clean_totals_ms, fit_table, why, fit_why —
        frozen into tp-plan.json by the caller.
    """
    degrees = list(range(1, 9))
    clean = {p: round(compute_ms / p + bytes_term_ms * (p - 1) / p, 1) for p in degrees}
    weights_gb, kv_gb, fixed_gb, budget_gb = 16.0, 4.0, 1.5, 15.0
    fits = {p: (weights_gb + kv_gb) / p + fixed_gb <= budget_gb for p in degrees}
    # Уровень A: почти написано. Впишите выводы своими словами и числами.
    predicted_knee_no_beat = 'TODO'  # int: степень минимума CLEAN-модели на 1..8
    fit_degree = 'TODO'              # int: минимальная степень, где fits[p] — True
    why = 'TODO: монотонна ли clean-модель и что это предсказывает для свипа?'
    fit_why = 'TODO: одно предложение — откуда взялась минимальная подходящая степень'
    return dict(
        predicted_knee_no_beat=predicted_knee_no_beat,
        fit_degree=fit_degree,
        clean_totals_ms=clean,
        fit_table={p: bool(v) for p, v in fits.items()},
        why=why, fit_why=fit_why)

_plan = plan_tp_knee(STEP_DECL['compute_ms'], STEP_DECL['bytes_term_ms'])
with open(os.path.join(RESULTS, 'tp-plan.json'), 'w', encoding='utf-8') as fh:
    json.dump(_plan, fh, ensure_ascii=False, indent=1)
print(f"tp-plan.json заморожен ДО свипа: предсказанное колено = {_plan['predicted_knee_no_beat']}, "
      f"минимальная подходящая степень = {_plan['fit_degree']}")

## Блок 1 — Механика: разбитое равно целому

Два способа резать матричное умножение (Megatron-style):

- **column-parallel**: режем `W` по **столбцам** (выходные признаки). Каждый GPU
  видит весь `X`, но пишет только свои столбцы `Y` — внутри блока стыковать не надо.
- **row-parallel**: режем `W` по **строкам**, а `X` — по столбцам на те же части.
  Каждый GPU даёт **частичную сумму**; соединяет их **all-reduce** (точная сумма).

Правило пары в MLP: вверх — column, вниз — row; на стыке — такт all-reduce.
В трансформер-блоке таких тактов два.

**Что доказываем числом:** сборка из шардов равна полному matmul с точностью до
float (≤ 1e-10). Это не иллюстрация — это проверка, которую вы напишете сами.

In [ ]:
# Учебный «блок»: два matmul (MLP: up-projection + down-projection) и вход X.
# Размеры нарочно маленькие: доказательство механики не требует больших тензоров.
import numpy as np

_D, _B = 24, 8
_rng = np.random.default_rng(SEED)
X_IN = _rng.normal(size=(_B, _D))
W_UP = _rng.normal(size=(_D, 4 * _D)) / np.sqrt(_D)     # column-parallel кандидат
W_DOWN = _rng.normal(size=(4 * _D, _D)) / np.sqrt(4 * _D)  # row-parallel кандидат
Y_REF = np.tanh(X_IN @ W_UP) @ W_DOWN  # полный (single-GPU) эталон
print(f'блок объявлен: X{_B}×{_D}, W_up {_D}×{4 * _D}, W_down {4 * _D}×{_D}; эталон Y готов')

In [ ]:
def column_parallel(x: np.ndarray, w: np.ndarray, parts: int) -> list:
    """Split the weight matrix by output columns; return per-shard outputs.

    Parameters
    ----------
    x : np.ndarray
        Input of shape (batch, features); every shard reads ALL of it.
    w : np.ndarray
        Weight of shape (features, out); split along axis=1.
    parts : int
        Shard count (the TP degree for this layer).

    Returns
    -------
    list of np.ndarray
        One output per shard: x @ w_j — its own columns of Y.
    """
    # Уровень B: скелет готов — заполните по шагам.
    # Шаг 1: np.array_split(w, parts, axis=1) — части по столбцам (выходные признаки).
    # Шаг 2: для каждой части верните x @ w_j.
    raise NotImplementedError('TODO: разрежьте w по столбцам и соберите выходы шардов')


_shards = column_parallel(X_IN, W_UP, 2)
print(f'column-parallel: {len(_shards)} шарда, форма выхода шарда {np.shape(_shards[0])}')

In [ ]:
def row_parallel(x: np.ndarray, w: np.ndarray, parts: int) -> list:
    """Split by input features; return PARTIAL products, not the joined output.

    Parameters
    ----------
    x : np.ndarray
        Input of shape (batch, features); split along axis=1.
    w : np.ndarray
        Weight of shape (features, out); split along axis=0, matching parts.
    parts : int
        Shard count.

    Returns
    -------
    list of np.ndarray
        Partial products x_j @ w_j — они ещё НЕ выход, их суммирует all-reduce.
    """
    # Шаг 1: np.array_split(x, parts, axis=1) и np.array_split(w, parts, axis=0).
    # Шаг 2: верните список x_j @ w_j (без суммирования!).
    raise NotImplementedError('TODO: частичные произведения — сумму делает allreduce_sum')


def allreduce_sum(partials: list) -> np.ndarray:
    """Join partial results with the exact all-reduce: elementwise sum.

    Parameters
    ----------
    partials : list of np.ndarray
        Same-shape partial products, one per shard.

    Returns
    -------
    np.ndarray
        The exact full output (within floating-point tolerance).
    """
    raise NotImplementedError('TODO: поэлементная сумма всех частичных результатов')

In [ ]:
# Доказательство: TP-сборка блока равна полному проходу с точностью до float.
# MLP по Megatron-схеме: вверх — column-parallel, вниз — row-parallel + all-reduce.
_rows_proof = []
for _parts in (1, 2, 4):
    _h = [np.tanh(h_j) for h_j in column_parallel(X_IN, W_UP, _parts)]
    _w2_parts = np.array_split(W_DOWN, _parts, axis=0)
    _y_tp = allreduce_sum([h_j @ w2_j for h_j, w2_j in zip(_h, _w2_parts)])
    _err = float(np.max(np.abs(_y_tp - Y_REF)))
    _rows_proof.append(dict(parts=_parts, max_abs_err=_err, within_tol=_err <= 1e-10))
with open(os.path.join(RESULTS, 'split-proof.json'), 'w', encoding='utf-8') as fh:
    json.dump(_rows_proof, fh, indent=1)
_all_ok = all(r['within_tol'] for r in _rows_proof)
print('доказано: ' + ', '.join(f"p={r['parts']}: err={r['max_abs_err']:.1e}" for r in _rows_proof)
      + (' — разбитое равно целому' if _all_ok else ' — ЕСТЬ РАСХОЖДЕНИЕ, стоп'))

## Справочник: термины блока

| Термин | Смысл |
| :-- | :-- |
| tp (degree) | число GPU, между которыми разрезаны веса одного слоя |
| column-parallel | разрез по выходным признакам; стыковка внутри блока не нужна |
| row-parallel | разрез по входным признакам; даёт частичные суммы |
| all-reduce | точная сумма частичных результатов на всех участниках |
| такт (beat) | одна синхронизация; в блоке их два, цена — `beat_ms` |
| straggler | самый медленный участник; шаг = max, не среднее |
| comm-bound | случай, когда синхронизация, а не вычисления, определяет шаг |

## Блок 2 — Экономика шага: свип по степеням

Полная объявленная модель шага:

`t(tp) = maxᵢ mᵢ · C/tp  +  B·(tp−1)/tp  +  beat·(tp−1)`

- `maxᵢ mᵢ · C/tp` — вычисления: делятся на tp, но straggler-множители
  `mᵢ ∈ [0.90, 1.15]` (seeded) забирают деление: шаг = самый медленный шард.
- `B·(tp−1)/tp` — байтовая доля all-reduce: растёт со степенью.
- `beat·(tp−1)` — цена тактов: то, чего НЕ было в вашем плане.

Вопрос свипа: где наблюдаемое колено и какой член его согнул? Сравнение — с
замороженным `tp-plan.json`, по ФОРМЕ (числа объявлены, не измерены).

In [ ]:
def declared_step_ms(degree: int, compute_ms: float, bytes_term_ms: float,
                     beat_ms: float, seed: int = SEED) -> dict:
    """Evaluate the declared step-time model for one TP degree.

    Parameters
    ----------
    degree : int
        TP degree (shard count) for this evaluation.
    compute_ms : float
        Declared compute part on a single GPU.
    bytes_term_ms : float
        Declared all-reduce byte share at (tp-1)/tp.
    beat_ms : float
        Declared per-beat synchronisation latency.
    seed : int, default SEED
        Deterministic straggler multipliers m_i ~ 1 + U(-0.10, 0.15), seeded
        by (seed, degree) so every run replays the same "hardware".

    Returns
    -------
    dict
        Row: degree, compute_ms (straggler-adjusted), share_ms, beats_ms,
        straggler (max multiplier), total_ms. Declared arithmetic — never a
        hardware measurement.
    """
    rng = np.random.default_rng((seed, degree))
    mult = 1.0 + rng.uniform(-0.10, 0.15, size=degree)
    compute = float(np.max(mult)) * compute_ms / degree
    share = bytes_term_ms * (degree - 1) / degree
    beats = beat_ms * (degree - 1)
    return dict(degree=degree,
                compute_ms=round(compute, 1),
                share_ms=round(share, 1),
                beats_ms=round(beats, 1),
                straggler=round(float(np.max(mult)), 3),
                total_ms=round(compute + share + beats, 1))


print('движок объявлен: пример p=4 →', declared_step_ms(4, **{
    'compute_ms': STEP_DECL['compute_ms'], 'bytes_term_ms': STEP_DECL['bytes_term_ms'],
    'beat_ms': STEP_DECL['beat_ms']})['total_ms'], 'мс (объявленная арифметика)')

In [ ]:
def sweep_tp(degrees: list, compute_ms: float, bytes_term_ms: float, beat_ms: float) -> list:
    """Journal one declared-engine row per TP degree.

    Parameters
    ----------
    degrees : list of int
        TP degrees to sweep, e.g. list(range(1, 9)).
    compute_ms, bytes_term_ms, beat_ms : float
        Declared knobs of the step-time model.

    Returns
    -------
    list of dict
        Rows as returned by declared_step_ms, one per degree.
    """
    # Уровень B: скелет готов — соберите строки по шагам.
    # Шаг 1: для каждой степени из degrees вызовите declared_step_ms(...).
    # Шаг 2: верните список строк (порядок degrees сохраняйте).
    raise NotImplementedError('TODO: свип по степеням через объявленный движок')


_rows_sweep = sweep_tp(list(range(1, 9)), STEP_DECL['compute_ms'],
                       STEP_DECL['bytes_term_ms'], STEP_DECL['beat_ms'])
print(f'свип: {len(_rows_sweep)} степеней')

In [ ]:
# График свипа: наблюдаемый total против чистой модели из плана (без цены такта).
import matplotlib
try:
    get_ipython()  # интерактивный фронтенд: инлайн-бэкенд, график виден
except NameError:
    matplotlib.use('Agg')  # headless-запуск: только файлы
import matplotlib.pyplot as plt

_degrees = [r['degree'] for r in _rows_sweep]
_totals = [r['total_ms'] for r in _rows_sweep]
_clean = [_plan['clean_totals_ms'][p] for p in _degrees]
_obs_knee = int(min(_rows_sweep, key=lambda r: r['total_ms'])['degree'])

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(_degrees, _totals, marker='o', color='#146957', label='свип (объявленная модель)')
ax.plot(_degrees, _clean, linestyle='--', color='#175cad', label='план: чистая модель (без тактов)')
ax.axvline(_obs_knee, color='#af372b', linestyle=':', linewidth=1.5,
           label=f'наблюдаемое колено p={_obs_knee}')
ax.set_xlabel('степень tensor parallelism (tp)')
ax.set_ylabel('мс на шаг (объявлено)')
ax.set_title('Шаг против степени: чистая модель зовёт «бери больше», такты говорят стоп')
ax.set_xticks(_degrees)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, 'sweep.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
def tp_verdict(plan: dict, rows: list) -> dict:
    """Compare the frozen plan with the sweep: observed knee and named mechanism.

    Parameters
    ----------
    plan : dict
        The frozen tp-plan.json content.
    rows : list of dict
        Sweep rows (degree/total_ms/...).

    Returns
    -------
    dict
        observed_knee, predicted_knee_no_beat, mechanism — frozen into verdict.json.
    """
    observed = min(rows, key=lambda r: r['total_ms'])['degree']
    # Уровень A: почти написано. Впишите механизм одним предложением.
    mechanism = 'TODO: какой член полной модели отсутствовал в чистой и куда он сдвинул колено?'
    return dict(
        observed_knee=observed,
        predicted_knee_no_beat=plan.get('predicted_knee_no_beat'),
        mechanism=mechanism)

_verdict = tp_verdict(_plan, _rows_sweep)
print('вердикт: ' + json.dumps(_verdict, ensure_ascii=False))

## Блок 3 — Ведомость памяти: какая степень влезает

На КАЖДЫЙ GPU при степени tp приходится:

- веса `16/tp` ГБ и KV `4/tp` ГБ — делятся (головы и слои разрезаны);
- `1.5` ГБ фиксированных (активации + framework) — **не** делятся;
- бюджет `15` ГБ — объявленный полезный объём одного ускорителя.

Вопрос: минимальная степень, где влезаем с запасом на KV? Ответ вы уже
заморозили в плане — теперь соберите ведомость построчно и сверьте.

In [ ]:
def memory_ledger(params_b: float, bytes_per_param: int, kv_gb: float,
                  fixed_gb: float, budget_gb: float, degree: int) -> dict:
    """Itemise one GPU's memory ledger at a TP degree and judge the fit.

    Parameters
    ----------
    params_b : float
        Parameter count (e.g. 8e9).
    bytes_per_param : int
        Bytes per parameter (2 for fp16).
    kv_gb, fixed_gb, budget_gb : float
        Declared KV cache, fixed per-GPU costs and the usable budget, GB.
    degree : int
        TP degree for this row.

    Returns
    -------
    dict
        Row: degree, weights_gb, kv_gb, fixed_gb, total_gb, fits (bool).
    """
    weights_gb = params_b * bytes_per_param / 1e9
    # Уровень B: скелет готов — заполните возврат по контракту.
    # weights и kv делятся на degree, fixed — нет; fits: total ≤ budget.
    raise NotImplementedError('TODO: ведомость одного GPU при данной степени')


_rows_ledger = [memory_ledger(MODEL_DECL['params_b'], MODEL_DECL['bytes_per_param'],
                              MODEL_DECL['kv_gb'], MODEL_DECL['fixed_gb'],
                              MODEL_DECL['budget_gb'], p) for p in range(1, 9)]
print(f'ведомость: {len(_rows_ledger)} строк')

In [ ]:
# Стековая диаграмма ведомости + линия бюджета: решение «влезает» читается глазом.
_fig, _ax = plt.subplots(figsize=(7.2, 3.4))
_degs = [r['degree'] for r in _rows_ledger]
_w = [r['weights_gb'] for r in _rows_ledger]
_k = [r['kv_gb'] for r in _rows_ledger]
_f = [r['fixed_gb'] for r in _rows_ledger]
_ax.bar(_degs, _w, color='#175cad', label='веса /tp')
_ax.bar(_degs, _k, bottom=_w, color='#146957', label='KV /tp')
_ax.bar(_degs, _f, bottom=[a + b for a, b in zip(_w, _k)], color='#89540a', label='фиксированное')
_ax.axhline(MODEL_DECL['budget_gb'], color='#af372b', linestyle='--', linewidth=1.5,
            label='бюджет 15 ГБ (объявлено)')
_ax.set_xlabel('степень tp')
_ax.set_ylabel('ГБ на один GPU')
_ax.set_title('Ведомость памяти: tp=1 не влезает — решение за строкой ledger, не за надеждой')
_ax.set_xticks(_degs)
_ax.legend(fontsize=8)
_fig.tight_layout()
_fig.savefig(os.path.join(RESULTS, 'ledger.png'), dpi=120, bbox_inches='tight')
plt.show()

## Блок 4 — Поломки как данные: два drill'а на стенограммах

Тот же жанр, что tag-drift и OOM в Теме 09: **стенограммы прошлых прогонов**
разбираются везде, железо не нужно. Два класса:

- **comm-bound**: степень подняли, compute упал, а шаг ВЫРОС — синхронизация
  съела выигрыш. Типичная цена «больше GPU» на медленной шине.
- **straggler**: один GPU в два раза медленнее — шаг равен САМОМУ медленному,
  не среднему. Деление на tp не спасает от дисбаланса.

Контракт разбора — cause / evidence / fix, уровень C: тело полностью ваше.

In [ ]:
# Стенограмма №1: wall-times по степеням из прошлого прогона (запись, не живые данные).
COMM_TABLE = dict(degree=[1, 2, 4, 8],
                  total_ms=[812.0, 489.0, 402.0, 451.0],
                  compute_ms=[798.0, 401.0, 203.0, 99.0])
# Стенограмма №2: времена шардов одного шага на 4 «GPU» (один в два раза медленнее).
STRAGGLER_TIMES = [101.0, 103.0, 99.0, 205.0]
print('стенограммы объявлены: comm-bound (4 степени) и straggler (4 шарда)')

In [ ]:
def diagnose_tp_drill(wall_by_degree: dict | None, shard_times_ms: list | None) -> dict:
    """Classify one tensor-parallel failure from its transcript.

    Parameters
    ----------
    wall_by_degree : dict or None
        {'degree': [...], 'total_ms': [...], 'compute_ms': [...]} for the
        comm-bound drill, or None when diagnosing the other class.
    shard_times_ms : list or None
        Per-shard times of one step (straggler drill), or None.

    Returns
    -------
    dict
        Keys cause, evidence, fix — the S5 record; numbers quoted from the
        transcript, not invented.
    """
    # Уровень C: контракт — docstring выше, тело полностью ваше.
    # Ожидаемые классы: comm-bound (total растёт, compute падает) и
    # straggler (max/median шардов > 1.5).
    raise NotImplementedError('TODO: классификация drill по контракту')

_rows_drills = [dict(drill='comm-bound (стенограмма)', **diagnose_tp_drill(COMM_TABLE, None)),
                dict(drill='straggler (стенограмма)', **diagnose_tp_drill(None, STRAGGLER_TIMES))]
print('разбор: 2 строки')

## Stretch — Карта осей и конфиг-карта vLLM

TP — не единственная ось. Одной строкой на ось (это S6):

- **TP** — режет матрицы ОДНОГО слоя; требует быстрой шины (NVLink); первый
  кандидат внутри узла, когда модель не влезает или нужен короткий шаг.
- **PP** — режет СЛОИ между устройствами; терпит медленные связи, платит
  «пузырями» конвейера.
- **DP** — реплицирует ЦЕЛИКОМ движок; сетевой обмен почти не нужен; ответ,
  когда модель влезает и нужен прежде всего пропуск.
- **EP** — режет ЭКСПЕРТЫ (MoE); каждый эксперт живёт на своём устройстве.

Реальный vLLM: одна ручка `tensor_parallel_size=N` — но оправдает ли её шина,
решает механика этого Notebook, а не флаг.

In [ ]:
PLACEMENT = dict(
    tp='TODO: что режет TP и когда он первый кандидат',
    pp='TODO: что режет PP и чем платит',
    dp='TODO: что реплицирует DP и когда него проще',
    ep='TODO: что режет EP и для каких моделей',
)
VLLM_CARD = dict(
    flag='tensor_parallel_size',
    value='TODO: минимальная степень, чтобы 16 ГБ весов + 4 ГБ KV влезли в 15 ГБ на GPU',
    precondition='TODO: что должно соединять GPU, чтобы TP оправдался',
)
_open = [k for k, v in {**PLACEMENT, **VLLM_CARD}.items() if str(v).startswith('TODO')]
if _open:
    print(f'заполните карту размещения: {_open}')
else:
    print('карта размещения и конфиг-карта vLLM готовы')

## Что вы построили — и чего эта лаба не доказывает

- Механику TP: разбитое равно целому (до 1e-10), пару column/row и два такта
  на блок — это ваше, доказано числом.
- План до свипа и честный вердикт: чистая модель звала «бери больше», такты
  и straggler поставили колено — механизм назван, не замечен.
- Ведомость памяти: минимальная подходящая степень — решение строки таблицы.
- Два класса поломок: comm-bound и straggler — cause/evidence/fix.

**Чего лаба НЕ доказывает:** ни одного числа о реальном GPU. Все миллисекунды —
объявленная арифметика; байты — учебный 8B-пример; реальные прогоны
`tensor_parallel_size≥2` на нескольких GPU — демонстрация владельца с отдельной
записью. Мост в Тему 11: сегодня вы умножили кухню по осям ВНУТРИ машины;
оркестровка таких машин по кластеру — Kubernetes.

In [ ]:
# Манифест попытки: sha256 всех артефактов растущего журнала.
artifacts = {}
for _name in ('tp-plan.json', 'split-proof.json', 'sweep.json', 'sweep.png',
              'verdict.json', 'ledger.json', 'ledger.png', 'drills.json'):
    _path = os.path.join(RESULTS, _name)
    if os.path.exists(_path):
        with open(_path, 'rb') as fh:
            artifacts[_name] = hashlib.sha256(fh.read()).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w', encoding='utf-8') as fh:
    json.dump(dict(profile=PROFILE, attempt=ATTEMPT, artifacts=artifacts),
              fh, ensure_ascii=False, indent=1)
print(f'{len(artifacts)} артефактов хэшировано в manifest.json — журнал попытки закрыт')